# Variational solution of the heat equation

Every implicit timestep of a PDE is a linear system, so a PDE solver is a linear
solver in a loop. This solves that system variationally instead of with phase
estimation — and runs into a limitation that is physics, not bookkeeping.

Companion to [`tutorials/03-pdes/02-variational-heat-equation.md`](../tutorials/03-pdes/02-variational-heat-equation.md).

**Requires:** `pip install -e ".[qiskit]"`

## 1. The timestep is a linear system

Discretising `du/dt = alpha d2u/dx2` with implicit Euler gives

```text
(I - alpha*dt*L) u_next = u_now
```

Implicit rather than explicit because it is unconditionally stable — explicit
would impose `dt < dx^2 / (2*alpha)` and make the timestep a stability question
rather than an accuracy one.

In [1]:
import numpy as np
from qprac_lab.algorithms.pdes.variational_heat_equation import heat_equation_system

matrix, initial, grid = heat_equation_system(num_qubits=3, alpha=0.1, dt=0.02)
print(f"{len(grid)} grid points (3 qubits), condition number {np.linalg.cond(matrix):.3f}")
print(f"initial profile: {np.round(initial, 4)}")

exact = np.linalg.solve(matrix, initial)
print(f"exact next step: {np.round(exact, 4)}")

8 grid points (3 qubits), condition number 1.364
initial profile: [0.     0.4339 0.7818 0.9749 0.9749 0.7818 0.4339 0.    ]
exact next step: [0.0351 0.4285 0.7672 0.9564 0.9564 0.7672 0.4285 0.0351]


## 2. The VQLS cost

Prepare `|psi(theta)>` with an ansatz and tune `theta` until `A|psi>` points along
`|b>`:

```text
C(theta) = 1 - |<b|A|psi>|^2 / <psi|A'A|psi>
```

Zero exactly when `A|psi>` is **parallel** to `|b>`. Parallel, not equal — and
that distinction is the whole caveat in section 4.

In [2]:
from qprac_lab.algorithms.pdes.variational_heat_equation import vqls_cost

target = initial / np.linalg.norm(initial)
print(f"cost at the true solution : {vqls_cost(matrix, target, exact/np.linalg.norm(exact)):.2e}")
rng = np.random.default_rng(0)
random_state = rng.normal(size=len(grid)); random_state /= np.linalg.norm(random_state)
print(f"cost at a random state    : {vqls_cost(matrix, target, random_state):.4f}")

cost at the true solution : 0.00e+00
cost at a random state    : 0.9532


## 3. Solve

Restarted by default: the cost is non-convex, so a single run reports one local
optimum — the same lesson the QAOA tutorial had to learn the hard way.

In [3]:
from qprac_lab.algorithms.pdes.variational_heat_equation import (
    run_variational_heat_equation_tutorial,
)

result = run_variational_heat_equation_tutorial()

print(f"{result.num_parameters} parameters, circuit depth {result.circuit_depth}")
print(f"VQLS cost {result.vqls_cost:.3e} after {result.function_evaluations} "
      f"evaluations across {result.restarts} restarts")
print(f"\nfidelity vs the exact direction: {result.fidelity:.9f}")
print(f"variational : {np.round(result.solution, 5)}")
print(f"exact       : {np.round(result.exact_solution, 5)}")

12 parameters, circuit depth 10
VQLS cost 1.344e-09 after 1308 evaluations across 5 restarts

fidelity vs the exact direction: 0.999999999
variational : [0.01912 0.23321 0.41752 0.52049 0.52052 0.41753 0.23321 0.01911]
exact       : [0.01911 0.23321 0.41753 0.5205  0.5205  0.41753 0.23321 0.01911]


Circuit depth **10**, against HHL's 86 for a smaller problem — and no
postselection, so every run counts. That is the variational trade: shallower
circuits, non-convex optimisation.

## 4. The caveat that is not a technicality

A quantum state is normalised. `|psi>` encodes the **direction** of `u`, never its
magnitude — and for the heat equation the magnitude *is* the physics.

In [4]:
tracking = result.norm_tracking
print(f"||u|| before the step        : {tracking['initial_norm']:.6f}")
print(f"||u|| after the step         : {tracking['exact_norm_after_step']:.6f}")
print(f"norm lost to diffusion       : {tracking['norm_lost_to_diffusion']:.6f}  <- the physics")
print(f"recovered from the state     : {tracking['recovered_from_quantum_state']}          <- none of it")

||u|| before the step        : 1.870829
||u|| after the step         : 1.837448
norm lost to diffusion       : 0.033381  <- the physics
recovered from the state     : 0.0          <- none of it


Heat leaving the system is the entire phenomenon being simulated, and the quantum
state cannot represent it. Any quantum PDE solver has to track that scale
classically, alongside.

This matters beyond bookkeeping: a demo reporting "fidelity 0.9999 against the
exact solution" while quietly normalising both sides is answering an easier
question than the one posed. The fidelity in section 3 is real, and so is the
zero in that last row.

## 5. What the scale carries

In [5]:
print(f"{'step':>5} {'time':>7} {'||u||':>10} {'peak':>10}")
print(f"{0:>5} {0.0:>7.2f} {tracking['initial_norm']:>10.6f} {initial.max():>10.6f}")
for step in result.steps:
    print(f"{step['step']:>5} {step['time']:>7.2f} {step['norm']:>10.6f} {step['peak']:>10.6f}")
print("\nMonotonic decay -- that curve is what a normalised state throws away.")

 step    time      ||u||       peak
    0    0.00   1.870829   0.974928
    1    0.02   1.837448   0.956386
    2    0.04   1.806974   0.938250
    3    0.06   1.778730   0.920545

Monotonic decay -- that curve is what a normalised state throws away.


## Classical baseline

`numpy.linalg.solve` on the same matrix: exact, and `O(N)` for a tridiagonal
system with a banded solver. The 8-point problem here takes microseconds, and the
circuit evaluations above bought a normalised copy of that answer.

## When not to use this

- **When you need the magnitude**, not just the shape — see section 4.
- **When a banded solver applies.** Tridiagonal systems are the easiest class of
  linear system there is.
- **Without restarts.** The cost is non-convex; one run is one local optimum.
- **At small `N`.** Everything here is faster and exact classically.

Compare with [HHL](05-hhl-linear-systems-intro.ipynb): phase estimation gives an
exact answer with no optimisation, at 8x the depth and with postselection
discarding most runs.